# Object Detection Workflow via API

**How to evaluate common workflows via the checkmaite API**

This notebook demonstrates a full end to end workflow of the checkmaite backend. The general steps are:

* Create wrapper objects as necessary (not all wrappers are needed for every tool)
  * Create a model wrapper
  * Create a dataset wrapper
  * Create a metric wrapper
* Create configuration to define analyses to run
* Generate "capability" object(s)
* Run analyses

All checkmaite wrappers are maite-compliant (https://mit-ll-ai-technology.github.io/maite/). 

**IMPORTANT**:   
* **Since we are using mini test datasets, the values presented in the results may not have significant meaning.** Use full datasets with models trained on similar data to view accurate results. 
* This notebook requires a dev install since it utilizes data from the test suite.

In [ ]:
from checkmaite import cache_path
from checkmaite._docs import create_expandable_output

import json
import logging
import os 
from pathlib import Path
import torch
import warnings

logger = logging.getLogger(__name__)

BASE_DIR = Path.cwd().parents[1]
EXAMPLE_DATA_DIR = BASE_DIR / "tests/data_for_tests"
EXAMPLE_COCO_DATASET_DIR = EXAMPLE_DATA_DIR.joinpath('coco_resized_val2017').resolve()
EXAMPLE_YOLO_DATASET_DIR = EXAMPLE_DATA_DIR.joinpath('yolo_dataset').resolve()
EXAMPLE_VISDRONE_DATASET_DIR = EXAMPLE_DATA_DIR.joinpath('visdrone_dataset')

warnings.filterwarnings("ignore", category=UserWarning)

### Set cache path 

Checkmaite has a default output cache path, but it can be overriden as needed. Note, the cache will only be written if the argument `use_prediction_and_evaluation_cache=True` is passed into the capability run.

In [ ]:
print(f'The default cache path is {cache_path()}')

# set the cache path
cache_path('tscache')

print(f'The current cache path is {cache_path()}')

### Define task 

In [ ]:
task = 'object_detection' 

### Create a model wrapper

Model wrappers hold the model object, weights and metadata.

Checkmaite has two object detection model wrappers - `torchvision` and `visdrone`. Below are examples of each of the available configuration options. 

#### Torchvision OD Model using default weights from Torchvision

In [ ]:
from checkmaite.core.object_detection.models import TorchvisionODModel
model_name = "ssdlite320_mobilenet_v3_large"
model_wrapper = TorchvisionODModel(model_name=model_name)

#### Torchvision OD Model using custom weights and config

Note: The configuration file is a JSON formatted text file that contains `index2label` as a key with its value being a dictionary of {index: category label} for the model e.g. `{0: '__background__', 1: 'person', 2: 'bicycle' ...}`.

In [ ]:
# save metadata and state_dict from previous cell to disk
config_path = cache_path() / "my_model.json"
pickle_path = cache_path() / "my_model.pt"

os.makedirs(os.path.dirname(config_path), exist_ok=True)
with open(config_path, "w") as f:
    json.dump({"index2label": model_wrapper.index2label}, f)
_ = torch.save(model_wrapper.model.state_dict(), pickle_path)

# extra kwargs to pass through to torchvision model object
kwargs = {}
torch_model_wrapper = TorchvisionODModel(
    model_name=model_name,
    weights_path=pickle_path,
    config_path=config_path,
    model_id="arbitraryidnumber",
    **kwargs,
)

#### VisDrone OD Model using weights from Kitware's data server

Visdrone architecture options are `resnet18`, `resnet50`, and `res2net50`. Custom weights on these models are not currently supported. 

In [ ]:
from checkmaite.core.object_detection.models import VisdroneODModel

# specify directory for the model weights to be downloaded
download_weights_dir = 'visdrone_model_weights'
visdrone_model_wrapper = VisdroneODModel(
    arch='resnet18',
    model_pickle_dir=download_weights_dir,
    model_id='1234',
)

### Create dataset wrapper

Dataset wrappers control the access to the dataset and contain metadata about the dataset and about individual images.

Checkmaite has three dataset wrappers for object detection - `coco`, `yolo` and `visdrone`. Below are examples of loading each type. 

#### COCO dataset

In [ ]:
from checkmaite.core.object_detection.dataset_loaders import CocoDetectionDataset

coco_dataset_wrapper = CocoDetectionDataset(
    root=str(EXAMPLE_COCO_DATASET_DIR),
    ann_file=str(EXAMPLE_COCO_DATASET_DIR.joinpath('instances_val2017_resized_6.json')),
)

coco_dataset_wrapper_single = CocoDetectionDataset(
    root=str(EXAMPLE_COCO_DATASET_DIR),
    ann_file=str(EXAMPLE_COCO_DATASET_DIR.joinpath('single_image.json')),
)

#### YOLO dataset

In [ ]:
from checkmaite.core.object_detection.dataset_loaders import YoloDetectionDataset

CURDIR = os.getcwd()
os.chdir(EXAMPLE_YOLO_DATASET_DIR.parents[2]) # the yolo test metadata file is written relative to the root pkg dir

yolo_dataset_wrapper = YoloDetectionDataset(
    yaml_dataset=str(EXAMPLE_YOLO_DATASET_DIR.joinpath('dataset.yaml')),
    ann_dir=str(EXAMPLE_YOLO_DATASET_DIR.joinpath('ann_dir')),
)
os.chdir(CURDIR)

#### Visdrone dataset 

In [ ]:
from checkmaite.core.object_detection.dataset_loaders import VisdroneDetectionDataset

visdrone_dataset_wrapper = VisdroneDetectionDataset(
    root=str(EXAMPLE_VISDRONE_DATASET_DIR),
)

### Create metric wrapper 

Metric wrappers provide standardized access to metric algorithms across the pydata ecosystem. 

Checkmaite has one object detection metric wrapper - `mean average precision (mAP)`. 

In [ ]:
from checkmaite.core.object_detection.metrics import map50_torch_metric_factory
metric_wrapper = map50_torch_metric_factory()
metric_wrapper.metadata['id'] = metric_wrapper.return_key

### Instantiate and run each Capability

As each capability is run, we collect slide data for Gradient PowePoint generation and add the results to our markdown report.

In [ ]:
# collect all the report slides in a list (to be convert to ppt after all stages are run)
slides = []
# Collect all the report MD strings in a single object (to be rendered and saved after all stages are executed)
report_md = ""

#### MAITE - Baseline evaluation 

In [ ]:
%%time
from checkmaite.core.object_detection import MaiteEvaluation

# instantiate capability object
capability = MaiteEvaluation()

# run analysis for this capability
maite_eval_run = capability.run(use_cache=False, datasets=[visdrone_dataset_wrapper], 
                               metrics=[metric_wrapper], models=[visdrone_model_wrapper])  # NOTE: `use_cache=True` will bypass the compute and load cached results if available

# view results
maite_eval_run.outputs

In [ ]:
# collect the markdown outputs for the final report
_md = maite_eval_run.collect_md_report(threshold=0.5)
report_md += _md

#### NRTK 

In [ ]:
%%time
from checkmaite.core.object_detection import NrtkRobustness, NrtkRobustnessConfig

nrtk_config_dict = {
      "name": "natural_robustness_",
      "perturber_factory": {
        "type": "nrtk.impls.perturb_image_factory.PerturberLinspaceFactory",
        "nrtk.impls.perturb_image_factory.PerturberLinspaceFactory": {
          "perturber": "nrtk.impls.perturb_image.photometric.enhance.BrightnessPerturber",
          "theta_key": "factor",
          "start": 0,
          "stop": 2,
          "num": 11
      }
   }
}

nrtk_config = NrtkRobustnessConfig(**nrtk_config_dict)
# instantiate capability object
capability = NrtkRobustness()

# run analysis for this capability
nrtk_run = capability.run(use_cache=False, datasets=[visdrone_dataset_wrapper], 
                               metrics=[metric_wrapper], models=[visdrone_model_wrapper], config=nrtk_config)

# view results
nrtk_run.outputs

In [ ]:
# collect the markdown outputs for the final report
_md = nrtk_run.collect_md_report(threshold=0.15)
report_md += _md

#### XAITK

XAITK should only be run on single images or a datasets with only a few images due to memory constraints

In [ ]:
%%time
from checkmaite.core.object_detection import XaitkExplainable, XaitkExplainableConfig

xaitk_config_dict = {
    'name': 'saliency_XAITKApp_0',
    'saliency_generator': {
        'type': 'xaitk_saliency.impls.gen_object_detector_blackbox_sal.drise.DRISEStack',
        'xaitk_saliency.impls.gen_object_detector_blackbox_sal.drise.DRISEStack': {
            'n': 20,
            's': 7,
            'p1': 0.7,
            'seed': 42,
            'threads': 8,
            'fill': [95, 96, 93],
        },
    },
    'img_batch_size': 1,
}
xaitk_config = XaitkExplainableConfig(**xaitk_config_dict)
# instantiate capability object
capability = XaitkExplainable()

# run analysis for this capability
xaitk_run = capability.run(use_cache=False, datasets=[coco_dataset_wrapper],
                           models=[visdrone_model_wrapper], config=xaitk_config)  # NOTE: `use_cache=True` will bypass the compute and load cached results if available

# view results
# output is large, so expandable outputs are shown instead of raw outputs here
create_expandable_output(xaitk_run.outputs.results)

In [ ]:
# collect the markdown outputs for the final report
_md = xaitk_run.collect_md_report(threshold=0)
report_md += _md

#### Dataeval - Bias

In [ ]:
%%time
from checkmaite.core.object_detection import DataevalBias

# instantiate capability object
capability = DataevalBias()

# run analysis for this test stage
bias_run = capability.run(use_cache=False, datasets=[coco_dataset_wrapper])  # NOTE: `use_cache=True` will bypass the compute and load cached results if available


In [ ]:
# collect the markdown outputs for the final report
_md = bias_run.collect_md_report(threshold=0)
report_md += _md

In [ ]:
# view results
# output is large, so expandable outputs are shown instead of raw outputs here
create_expandable_output(bias_run.outputs.balance)

In [ ]:
create_expandable_output(bias_run.outputs.coverage)

In [ ]:
create_expandable_output(bias_run.outputs.diversity)

#### Dataeval - Feasibility

NOTE: Currently non-functional ([checkmaite Issue #181](https://gitlab.jatic.net/jatic/reference-implementation/reference-implementation/-/issues/181))

In [ ]:
# %%time
# from checkmaite.object_detection.test_stages import DatasetFeasibilityTestStage

# # instantiate test stage object 
# test_stage = DatasetFeasibilityTestStage()
# # populate test_stage with data
# test_stage.load_dataset(visdrone_dataset_wrapper, dataset_id='dataset1')
# test_stage.load_threshold(0.5)

# # run analysis for this test stage
# feasibility_run = test_stage.run(use_stage_cache=False)  # NOTE: `use_stage_cache=True` will bypass the compute and load cached results if available

# # collect the slides for the final report
# stage_slides = test_stage.collect_report_consumables()
# # add to overall slide list
# slides += stage_slides

# # view results
# feasibility_run.outputs

#### Dataeval - Cleaning

In [ ]:
%%time
from checkmaite.core.object_detection import DataevalCleaning

# instantiate capability object
capability = DataevalCleaning()

# run analysis for this capability
cleaning_run = capability.run(use_cache=False, datasets=[visdrone_dataset_wrapper])  # NOTE: `use_cache=True` will bypass the compute and load cached results if available

# view results
create_expandable_output(cleaning_run.outputs)

In [ ]:
# collect the markdown outputs for the final report
_md = cleaning_run.collect_md_report(threshold=0)
report_md += _md

#### Dataeval - Shift

In [ ]:
%%time
from checkmaite.core.object_detection import DataevalShift

# instantiate capability object
capability = DataevalShift()

# run analysis for this capability
shift_run = capability.run(use_cache=False, datasets=[coco_dataset_wrapper, coco_dataset_wrapper])  # NOTE: `use_cache=True` will bypass the compute and load cached results if available

# view results
# output is large, so expandable outputs are shown instead of raw outputs here
create_expandable_output(shift_run.outputs)

In [ ]:
# collect the markdown outputs for the final report
_md = shift_run.collect_md_report(threshold=0)
report_md += _md

## Construct final report

Finally, we build our reports using the collected capability run outputs.

Below we include cells for both generating PowerPoint and HTML-based reports using the external Gradient capability, and a Markdown-based report using a package-native rendering feature.

In [ ]:
from checkmaite.core.report._markdown import create_markdown_output

# construct MD report with summarized results
report_path = cache_path() / "report"
report_path.mkdir(parents=True, exist_ok=True)
report_filename = 'RI_OD_Sample_Report.md'
report = create_markdown_output(report_md, report_path, report_filename)